# 02 · Deck legality

`crdeck.rules` is the only code that decides legality. Slot rules come from `config/deck_rules.yaml`.
Unresolved rules (two champions? what the Wild slot accepts?) are config values, defaulting to the safe reading.

In [ ]:
import os
import json
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

DATA_SOURCE = os.environ.get("CRDECK_DATA_SOURCE", "real")  # "real" (default) or "synthetic"
from crdeck.config import Paths
paths = Paths.for_source(DATA_SOURCE)
print(paths.banner())

In [ ]:
from crdeck.rules import DeckRules, DeckCard, check_deck, check_partial
from crdeck.cards import load_card_table
rules = DeckRules.load()
print(rules)
try:
    cards = load_card_table(paths.processed / "cards.parquet")
    print("card table: API ids from", paths.processed / "cards.parquet")
except FileNotFoundError:
    # Legality depends only on card flags, not on ids, so placeholder ids are fine for this demo.
    from crdeck.synthetic import synthetic_card_table
    cards = synthetic_card_table()
    print("card table: hand labels with PLACEHOLDER ids (no /cards response yet); fine for legality demos only")
C = lambda name, form="base": DeckCard(cards.by_name(name).card_id, form)

In [ ]:
base = ["Hog Rider", "Fireball", "The Log", "Ice Spirit", "Skeletons", "Cannon"]
examples = {
    "two evolutions + one hero": [C(n) for n in base[:5]] + [C("Archers", "evo"), C("Bats", "evo"), C("Musketeer", "hero")],
    "three evolutions": [C(n) for n in base[:5]] + [C("Archers", "evo"), C("Bats", "evo"), C("Zap", "evo")],
    "champion + evo + hero": [C(n) for n in base[:5]] + [C("Archer Queen"), C("Archers", "evo"), C("Knight", "hero")],
    "two champions": [C(n) for n in base] + [C("Archer Queen"), C("Golden Knight")],
    "same card base + evo": [C(n) for n in base] + [C("Knight"), C("Knight", "evo")],
    "nine cards": [C(n) for n in base] + [C("Knight"), C("Valkyrie"), C("Goblins")],
    "Hog Rider evolution (no such form)": [C(n) for n in base[1:]] + [C("Hog Rider", "evo"), C("Knight"), C("Valkyrie")],
}
rows = []
for name, d in examples.items():
    r = check_deck(d, cards, rules)
    rows.append({"case": name, "legal": r.legal, "reasons": ", ".join(r.reasons), "slots": {cards.name(DeckCard.parse(k).card_id) + ":" + DeckCard.parse(k).form: s for k, s in r.slots.items()}})
pd.DataFrame(rows)

`max_champions` is a config value; both settings are tested:

In [ ]:
d = examples["two champions"]
print("max_champions=1:", check_deck(d, cards, rules).legal, "| max_champions=2:", check_deck(d, cards, rules.with_max_champions(2)).legal)
print(check_partial([C("Archer Queen"), C("Golden Knight"), C("Mighty Miner")], cards, rules).details)

## Do the rules match real play?
The slot rules are checked against every deck played in usable battles. Slots come from deck positions (observed: 0 = Evolution, 1 = Hero, 2 = Wild). Any deck the validator rejects means `config/deck_rules.yaml` disagrees with the game.

In [ ]:
from crdeck.evaluate import observed_deck_audit
try:
    audit = observed_deck_audit(paths, cards, rules)
except FileNotFoundError as e:
    audit = None
    print("No collected battles yet:", e)
if audit:
    print("distinct decks:", audit["distinct_decks"], "| rejected by validator:", audit["rejected_by_validator"] or "none")
    print("champion slots:", audit["champion_slots"], "| evo slots:", audit["evo_slots"], "| hero slots:", audit["hero_slots"])
    print("base cards in special slots:", audit["base_cards_in_special_slots"])
    display(audit["combinations"])

## Legality tests

In [ ]:
import subprocess, sys
from crdeck.config import PROJECT_ROOT
out = subprocess.run([sys.executable, "-m", "pytest", "-q", str(PROJECT_ROOT / "tests" / "test_rules.py")], capture_output=True, text=True)
print(out.stdout[-600:])
assert out.returncode == 0, "legality tests failing: do not move on (spec 11)"